## Competition link:<a href="https://www.kaggle.com/competitions/house-prices-advanced-regression-techniques/overview">House Prices - Advanced Regression Techniques</a>
## Created by <a href="https://github.com/yunsuxiaozi">yunsuxiaozi</a> 2025/09/27

# <span><h1 style = "font-family: garamond; font-size: 40px; font-style: normal; letter-spcaing: 3px; background-color: #f6f5f5; color :#fe346e; border-radius: 100px 100px; text-align:center">1.Import Libraries</h1></span>

Due to the continuous updates of the <a href="https://github.com/yunsuxiaozi/Yunbase">Yunbase</a> GitHub repository, a static version is used <a href="https://www.kaggle.com/code/yunsuxiaozi/yunbase">here</a>.We are using version 24 here.

This is a framework I developed for the convenience of playing competitions and avoiding writing duplicate code. You can simply understand it as automl, although its current performance is not as good as automl.

If you are not familiar with `Yunbase`, it's okay. Yunbase mainly encapsulates some commonly used operations such as `onehotencoder`、`kfold`, You can also manually implement the same function without using Yunbase. This notebook mainly provides beginners with some ideas for feature engineering and cross validation.

If there is a long time, such as a month or even a year, after the release date of this notebook, this code may not be able to run directly.Since the version of `Yunbase` will be constantly updated, if you want to run this code successfully, you might first download the output from the corresponding version in <a href="https://www.kaggle.com/code/yunsuxiaozi/yunbase">Yunbase</a>, save it as your own dataset, and then import it.

## About pytabkit

Yunbase currently supports five models: lightgbm, xgboost, catboost, realmlp, and tabm. It can also support other models that provide fit and predict interfaces, such as linear regression, but other models may have compatibility issues.

The use of realmlp and tabm can be done using the pytabkit library, which provides interfaces for fit and predict

In [ ]:
!pip -q install -U pytabkit

In [ ]:
import sys
sys.path.append('/kaggle/input/yunbase/Yunbase')

In [ ]:
!pip install -q --requirement /kaggle/input/yunbase/Yunbase/requirements.txt  \
--no-index --find-links file:/kaggle/input/yunbase/

In [ ]:
from baseline import Yunbase
import pandas as pd#read csv,parquet
import numpy as np#for scientific computation of matrices
from  lightgbm import LGBMRegressor,LGBMClassifier,log_evaluation,early_stopping
from catboost import CatBoostRegressor,CatBoostClassifier
from pytabkit import RealMLP_TD_Classifier,RealMLP_TD_Regressor
from xgboost import XGBRegressor,XGBClassifier
from pytabkit import TabM_D_Classifier,TabM_D_Regressor
import warnings#avoid some negligible errors
#The filterwarnings () method is used to set warning filters, which can control the output method and level of warning information.
warnings.filterwarnings('ignore')

import random#provide some function to generate random_seed.
#set random seed,to make sure model can be recurrented.
def seed_everything(seed):
    np.random.seed(seed)#numpy's random seed
    random.seed(seed)#python built-in random seed
seed_everything(seed=2025)

# <span><h1 style = "font-family: garamond; font-size: 40px; font-style: normal; letter-spcaing: 3px; background-color: #f6f5f5; color :#fe346e; border-radius: 100px 100px; text-align:center">2.Load data</h1></span>

In [ ]:
train=pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test=pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")
CATS=[c for c in train.columns if train[c].dtype==object]
print(f"train.shape:{train.shape},test.shape:{test.shape}")
train.head()

By observing this dataset, it can be found that the data volume is very small. For such data, we generally do not perform too much feature engineering to prevent overfitting. We can ensemble multiple models to achieve good results.

# <span><h1 style = "font-family: garamond; font-size: 40px; font-style: normal; letter-spcaing: 3px; background-color: #f6f5f5; color :#fe346e; border-radius: 100px 100px; text-align:center">3.Model training</h1></span>

Passing in some basic parameters can complete the training of the model.

In [ ]:
models=[ (TabM_D_Regressor(weight_decay=0.05),'tabm'),
         (LGBMRegressor(device='gpu',gpu_use_dp=True),'lgb'),
         (RealMLP_TD_Regressor(),'realmlp'),
         (CatBoostRegressor(task_type='GPU'),'cat'),
         (XGBRegressor(tree_method='gpu_hist',enable_categorical=True),'xgb')
       ] 

yunbase=Yunbase(num_folds=10,
                  models=models,
                  objective='regression',
                  metric='rmsle',
                  one_hot_max=-1,
                  target_col='SalePrice',
                  exp_mode=True,
                  device='gpu',
)
yunbase.fit(train,category_cols=CATS,
            save_trained_models=False)

# <span><h1 style = "font-family: garamond; font-size: 40px; font-style: normal; letter-spcaing: 3px; background-color: #f6f5f5; color :#fe346e; border-radius: 100px 100px; text-align:center">4.Create submission</h1></span>

In [ ]:
test_preds=yunbase.predict(test)
yunbase.submit("/kaggle/input/house-prices-advanced-regression-techniques/sample_submission.csv",test_preds=test_preds)